# Workflow description
Using ensemble of 2 xml-roberta models: ["joeddav/xlm-roberta-large-xnli", "symanto/xlm-roberta-base-snli-mnli-anli-xnli"]

Find best weights on train dataset

In [1]:
import torch
import pandas as pd
from tqdm import tqdm
import keras_nlp, keras_hub
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.model_selection import train_test_split
import numpy as np

2026-04-06 18:17:43.357358: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775499463.604696      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775499463.689532      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775499464.281590      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775499464.281630      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775499464.281633      24 computation_placer.cc:177] computation placer alr

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load Models and Tokenizers
model_names = ["joeddav/xlm-roberta-large-xnli", "symanto/xlm-roberta-base-snli-mnli-anli-xnli"]
tokenizers = [AutoTokenizer.from_pretrained(model_name) for model_name in model_names]
models = [AutoModelForSequenceClassification.from_pretrained(model_name).to(device) for model_name in model_names]
# Set to evaluation mode
for model in models:
    model.eval()


# 3. Prepare Dataset Class
class NLIDataset(Dataset):
    def __init__(self, df):
        self.premises = df['premise'].astype(str).tolist()
        self.hypotheses = df['hypothesis'].astype(str).tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.premises[idx], self.hypotheses[idx], self.labels[idx]


# Load your data
base_dir = "/kaggle/input/competitions/contradictory-my-dear-watson"
train_df = pd.read_csv(f'{base_dir}/train.csv')
test_df = pd.read_csv(f'{base_dir}/test.csv')
train_subset, val_subset = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

print(f"Train size: {len(train_subset)}, Val size: {len(val_subset)}")

train_dataset = NLIDataset(train_subset)
# Adjust batch_size based on your GPU memory (VRAM)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

# 5. Find best weights
all_preds = [[], []]
all_labels = []
for idx, (model, tokenizer) in enumerate(zip(models, tokenizers)):
    with torch.no_grad():  # Disable gradient calculation for speed/memory
        for premises, hypotheses, labels in tqdm(train_loader, desc="Inference"):
            # Tokenize batch and move to GPU
            inputs = tokenizer(
                list(premises),
                list(hypotheses),
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(device)

            # Forward pass
            outputs = model(**inputs)

            # Get predictions (highest logit index)
            preds = torch.softmax(outputs.logits, dim=1)

            all_preds[idx].extend(preds.cpu().numpy())
            if idx == 0:
                all_labels.extend(labels)

# 5. Find best weights
labels_map_kaggle = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
for idx, model in enumerate(models):
    reorder_idx = [labels_map_kaggle[class_name.lower()] for class_name in model.config.id2label.values()]
    all_preds[idx] = [logits[reorder_idx] for logits in all_preds[idx]]


weight_pairs = [(p, 1 - p) for p in np.arange(0, 1.05, 0.05)]
best_weights = ()
best_accuracy = 0
for w1, w2 in weight_pairs:
    preds = []
    for idx in range(len(all_preds[0])):
        preds.append(np.argmax((all_preds[0][idx]*w1 + all_preds[1][idx]*w2)/2))
    correct = sum(1 for p, l in zip(preds, all_labels) if p == l)
    accuracy = (correct / len(all_labels)) * 100
    print(f"({w1},{w2}): {accuracy})")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_weights = w1,w2

print(f"Best weights: {best_weights}")
print(f"Best accuracy: {best_accuracy}")

# --- SUBMISSION GENERATION ---
print("\n--- Starting Inference on Test Set ---")


# Use a specific Dataset class for test (no labels)
class NLITestDataset(Dataset):
    def __init__(self, df):
        self.ids = df['id'].tolist()
        self.premises = df['premise'].astype(str).tolist()
        self.hypotheses = df['hypothesis'].astype(str).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], self.premises[idx], self.hypotheses[idx]


test_loader = DataLoader(NLITestDataset(test_df), batch_size=32, shuffle=False)

# 2. Collect Softmax Predictions for both models
test_preds = [[], []]
test_ids = []

for idx, (model, tokenizer) in enumerate(zip(models, tokenizers)):
    model.eval()
    # Reorder index to align model labels with Kaggle labels (0: entailment, 1: neutral, 2: contradiction)
    reorder_idx = [labels_map_kaggle[model.config.id2label[i].lower()] for i in range(3)]

    with torch.no_grad():
        for ids, premises, hypotheses in tqdm(test_loader, desc=f"Model {idx + 1} Test Inference"):
            inputs = tokenizer(
                list(premises),
                list(hypotheses),
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            # Apply softmax and reorder columns to match Kaggle format
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            test_preds[idx].extend(probs[:, reorder_idx])

            if idx == 0:
                test_ids.extend(ids)

# 3. Apply Best Weights for Ensemble
w1, w2 = best_weights
final_predictions = []

for i in range(len(test_ids)):
    # Weighted average of the probabilities
    weighted_prob = (test_preds[0][i] * w1) + (test_preds[1][i] * w2)
    final_predictions.append(np.argmax(weighted_prob))

# 4. Create and Save Submission
submission = pd.DataFrame({
    'id': test_ids,
    'prediction': final_predictions
})

print("\nSubmission preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Prediction distribution:\n{submission['prediction'].value_counts()}")

submission.to_csv('submission.csv', index=False)
print("\n✓ submission.csv saved and ready for Kaggle submission!")
# Ensemble 93.34777227722772 weights 0.5, 0.5

Using device: cuda


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Train size: 9696, Val size: 2424



Inference: 100%|██████████| 152/152 [01:00<00:00,  2.50it/s]


(0.0,1.0): 89.51113861386139)
(0.05,0.95): 89.79991749174917)
(0.1,0.9): 90.0990099009901)
(0.15000000000000002,0.85): 90.41872937293729)
(0.2,0.8): 90.75907590759076)
(0.25,0.75): 91.10973597359737)
(0.30000000000000004,0.7): 91.48102310231023)
(0.35000000000000003,0.6499999999999999): 91.86262376237624)
(0.4,0.6): 92.36798679867987)
(0.45,0.55): 92.89397689768977)
(0.5,0.5): 93.2240099009901)
(0.55,0.44999999999999996): 93.00742574257426)
(0.6000000000000001,0.3999999999999999): 92.92491749174917)
(0.65,0.35): 92.79084158415841)
(0.7000000000000001,0.29999999999999993): 92.72896039603961)
(0.75,0.25): 92.63613861386139)
(0.8,0.19999999999999996): 92.54331683168317)
(0.8500000000000001,0.1499999999999999): 92.50206270627062)
(0.9,0.09999999999999998): 92.42986798679867)
(0.9500000000000001,0.04999999999999993): 92.4092409240924)
(1.0,0.0): 92.4092409240924)
Best weights: (np.float64(0.5), np.float64(0.5))
Best accuracy: 93.2240099009901

--- Starting Inference on Test Set ---


Model 2 Test Inference: 100%|██████████| 163/163 [00:29<00:00,  5.51it/s]


Submission preview:
           id  prediction
0  c6d58c3f69           2
1  cefcc82292           1
2  e98005252c           0
3  58518c10ba           1
4  c32b0d16df           1

Submission shape: (5195, 2)
Prediction distribution:
prediction
0    1866
1    1720
2    1609
Name: count, dtype: int64

✓ submission.csv saved and ready for Kaggle submission!
